# PathSnap — Dataset Exploration (Person 1)
This notebook inspects the supplied training and student-test trip files. It preserves original trip and arc IDs and does not use evaluator data.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np
ROOT = Path('../data')  # run notebook from notebooks/

In [ ]:
sys.path.insert(0, str(ROOT.parent / 'src'))
from data.loader import iter_trips
from data.validation import validate_trip

In [ ]:
rows=[]; checks=[]
for partition in ['train','student_test']:
    for d in iter_trips(ROOT, partition):
        t,n,a,r = (d[k] for k in ['track','nodes','arcs','route'])
        dt=np.diff(t[:,2])
        rows.append({'trip_id':d['trip_id'],'partition':partition,'gps_points':len(t),'nodes':len(n),'arcs':len(a),'route_entries':len(r),'max_gap_s':dt.max() if len(dt) else 0,'gaps_over_10s':int((dt>10).sum()),'duplicate_track_rows':int(pd.DataFrame(t).duplicated().sum())})
        checks.append(validate_trip(d))
inventory=pd.DataFrame(rows)
inventory.to_csv('../results/dataset_inventory.csv',index=False)
inventory.groupby('partition').agg(['count','sum','mean'])

In [ ]:
pd.DataFrame(checks).query('valid == False')

## Notes
The `.route` file contains ordered arc IDs but no direct GPS observation alignment. The current candidate labels therefore use the nearest geometric arc among the supplied route arcs as a provisional association. Candidate miss rate is a proxy, not official route accuracy.